# Milestone 5 — Isolation Forest Model Development
## AI Bill Anomaly Detection System

**Why Isolation Forest:**
- Unsupervised — no labeled fraud dataset needed (we only have `True_Anomaly` because
  we *simulated* the data; a real deployment wouldn't have it)
- Efficient on high-dimensional tabular data
- Naturally outputs a continuous anomaly score, not just a binary label — this is what
  lets us build a graded risk score in Milestone 6, instead of a blunt yes/no

**Important discipline for this notebook:** `True_Anomaly` is loaded **only for a final
sanity check at the end** — never as a training input. Everything up to that point is
trained the way it would be in a real, unlabeled deployment.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/processed/invoices_features.csv", parse_dates=["Invoice_Date"])

with open("../data/processed/model_feature_columns.json") as f:
    model_feature_cols = json.load(f)

print("Dataset shape:", df.shape)
print("Number of model features:", len(model_feature_cols))

Dataset shape: (10200, 43)
Number of model features: 34


## 1. Prepare training data

In [2]:
X = df[model_feature_cols].copy()

# Isolation Forest doesn't strictly require scaling (it splits on raw thresholds, not
# distances), but we scale anyway for two reasons: (1) it makes feature magnitudes
# comparable if we ever swap in a distance-based model later, and (2) it's good practice
# to demonstrate in a portfolio project.
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

print("Feature matrix shape:", X_scaled.shape)
X_scaled.describe().T[["mean", "std", "min", "max"]].head()

Feature matrix shape: (10200, 34)


,mean,std,min,max
Amount,-7.767207e-17,1.000049,-0.803770,9.795561
GST,-1.027501e-17,1.000049,-1.655740,4.320392
Quantity,8.463818e-17,1.000049,-1.701427,1.700135
Invoice_Month,-9.822208e-17,1.000049,-1.601362,1.585174
Invoice_Weekday,8.777293e-17,1.000049,-1.520428,1.490807


## 2. Train Isolation Forest

We train two versions to compare:
1. **`contamination="auto"`** — what you'd use in a real deployment, where you don't
   know the true anomaly rate in advance.
2. **`contamination=0.0825`** — informed by our EDA (Milestone 3), since we know the
   true injected rate here. We compare both to see how much the assumed contamination
   rate actually matters.

In [3]:
iso_auto = IsolationForest(
    n_estimators=200,
    contamination="auto",
    max_samples="auto",
    random_state=42,
    n_jobs=-1
)
iso_auto.fit(X_scaled)

iso_informed = IsolationForest(
    n_estimators=200,
    contamination=0.0825,
    max_samples="auto",
    random_state=42,
    n_jobs=-1
)
iso_informed.fit(X_scaled)

print("Both models trained.")

Both models trained.


## 3. Generate predictions and anomaly scores

In [4]:
# decision_function: higher = more normal, lower/negative = more anomalous
# predict: -1 = anomaly, 1 = normal (sklearn convention)

df["anomaly_raw_auto"] = iso_auto.decision_function(X_scaled)
df["Predicted_Anomaly_auto"] = (iso_auto.predict(X_scaled) == -1).astype(int)

df["anomaly_raw_informed"] = iso_informed.decision_function(X_scaled)
df["Predicted_Anomaly_informed"] = (iso_informed.predict(X_scaled) == -1).astype(int)

print("Auto contamination -> predicted anomalies:", df["Predicted_Anomaly_auto"].sum(),
      f"({df['Predicted_Anomaly_auto'].mean()*100:.2f}%)")
print("Informed contamination -> predicted anomalies:", df["Predicted_Anomaly_informed"].sum(),
      f"({df['Predicted_Anomaly_informed'].mean()*100:.2f}%)")
print("\nTrue anomaly rate (ground truth, for reference only):",
      f"{df['True_Anomaly'].mean()*100:.2f}%")

Auto contamination -> predicted anomalies: 534 (5.24%)
Informed contamination -> predicted anomalies: 842 (8.25%)

True anomaly rate (ground truth, for reference only): 8.25%


## 4. Quick sanity check against ground truth

In [5]:
from sklearn.metrics import classification_report, confusion_matrix

for label, col in [("auto", "Predicted_Anomaly_auto"), ("informed", "Predicted_Anomaly_informed")]:
    print(f"=== contamination={label} ===")
    print(confusion_matrix(df["True_Anomaly"], df[col]))
    print(classification_report(df["True_Anomaly"], df[col], target_names=["Normal", "Anomaly"]))
    print()

=== contamination=auto ===
[[9326   32]
 [ 340  502]]
              precision    recall  f1-score   support

      Normal       0.96      1.00      0.98      9358
     Anomaly       0.94      0.60      0.73       842

    accuracy                           0.96     10200
   macro avg       0.95      0.80      0.86     10200
weighted avg       0.96      0.96      0.96     10200


=== contamination=informed ===
[[9087  271]
 [ 271  571]]
              precision    recall  f1-score   support

      Normal       0.97      0.97      0.97      9358
     Anomaly       0.68      0.68      0.68       842

    accuracy                           0.95     10200
   macro avg       0.82      0.82      0.82     10200
weighted avg       0.95      0.95      0.95     10200




**Note:** full precision/recall analysis with ROC/PR curves and threshold tuning belongs
in **Milestone 6 (Risk Scoring & Evaluation)** — this is just a sanity check to confirm the
model is directionally working before we move on. We proceed with the **informed
contamination model** as the primary model, since it's calibrated to what EDA told us about
this dataset's anomaly rate — but we keep both `anomaly_raw` columns for comparison in
Milestone 6.

## 5. Feature importance (approximate, via permutation)

In [6]:
# Isolation Forest has no native feature_importances_, and sklearn's permutation_importance
# requires a supervised scorer (needs y), which we don't have. So we compute a manual
# version instead: how much does shuffling each column change decision_function output?
importances = {}
baseline = iso_informed.decision_function(X_scaled)
for col in X_scaled.columns:
    X_shuffled = X_scaled.copy()
    X_shuffled[col] = np.random.RandomState(42).permutation(X_shuffled[col].values)
    shuffled_scores = iso_informed.decision_function(X_shuffled)
    importances[col] = np.mean(np.abs(baseline - shuffled_scores))

importance_series = pd.Series(importances).sort_values(ascending=False)
importance_series.head(10)

Category_Electronics              0.007657
Category_Construction Material    0.006882
Category_Fuel                     0.006636
Department_It & Electronics       0.006538
Department_Water Supply           0.006190
Department_Sanitation             0.006006
Category_Medical Supplies         0.005989
Category_Vehicle Maintenance      0.005960
Category_Stationery               0.005799
Department_Education              0.005617
dtype: float64

**Actual result — worth understanding, not just reporting:** the ranking is dominated
by one-hot `Category_*` / `Department_*` dummy columns, with `Amount_Deviation` and
`Vendor_Frequency` sitting in the *middle* of the ranking rather than at the top. That's
not what EDA would suggest, and it's a methodology artifact worth flagging rather than
hiding:

- After `StandardScaler`, a sparse binary column (e.g. a category that's only 10% of
  rows) gets scaled to a few extreme values for the minority class. Shuffling that column
  swaps those extreme values into many more rows, which produces a large swing in
  `decision_function` — inflating its apparent "importance" — even though it isn't
  necessarily what drives the actual anomaly splits.
- This is a known weakness of naive permutation importance on one-hot encoded, imbalanced
  categorical features. It doesn't mean the engineered features are useless — the
  classification report above already shows the model is finding real signal (68%
  precision/recall on the informed-contamination version, well above the ~8% base rate).
- A more reliable read on feature contribution would use **SHAP values** (listed in the
  original Future Scope), which isn't in this milestone's build but is a good thing to
  mention if asked "how would you improve this" in an interview.

## 6. Save model, scaler, and scored dataset

In [7]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(iso_informed, "../models/isolation_forest.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

with open("../models/model_feature_columns.json", "w") as f:
    json.dump(model_feature_cols, f, indent=2)

print("Saved: models/isolation_forest.pkl")
print("Saved: models/scaler.pkl")
print("Saved: models/model_feature_columns.json")

Saved: models/isolation_forest.pkl
Saved: models/scaler.pkl
Saved: models/model_feature_columns.json


In [8]:
output_path = "../data/processed/invoices_scored.csv"
df.to_csv(output_path, index=False)
print(f"Saved scored dataset to {output_path}")
print(f"Shape: {df.shape}")
df[["Invoice_ID", "Amount", "Amount_Deviation", "anomaly_raw_informed",
    "Predicted_Anomaly_informed", "True_Anomaly"]].sample(5)

Saved scored dataset to ../data/processed/invoices_scored.csv
Shape: (10200, 47)


,Invoice_ID,Amount,Amount_Deviation,anomaly_raw_informed,Predicted_Anomaly_informed,True_Anomaly
1942,INV101612,21409.34,0.792019,0.030903,0,0
4010,INV107846,195643.66,9.217716,-0.103755,1,1
4280,INV102226,10112.26,0.891985,0.014303,0,0
7773,INV104998,21440.05,0.920085,0.005367,0,0
7665,INV108960,14373.62,0.677211,0.060905,0,0


## Summary

| Item | Value |
|---|---|
| Model | Isolation Forest, 200 estimators |
| Contamination used for primary model | 0.0825 (informed by EDA) |
| Features used | 34 (see `model_feature_columns.json`) |
| Top permutation-importance features | Sparse `Category_*`/`Department_*` dummies (likely a scale artifact — see note above) |
| Key engineered signal (per Milestone 4 validation) | `Amount_Deviation` (0.81 normal vs. 3.07 anomalous, from Milestone 4) |
| Artifacts saved | `models/isolation_forest.pkl`, `models/scaler.pkl` |

## Notes for Milestone 6 (Risk Scoring & Evaluation)

- Load `data/processed/invoices_scored.csv`
- Convert `anomaly_raw_informed` (continuous, sklearn convention: lower = more anomalous)
  into a **0–100 risk score** (higher = more risky) via min-max normalization + inversion
- Bucket into Low / Medium / High per the original spec (0–30 / 31–70 / 71–100)
- Run full precision/recall/ROC evaluation against `True_Anomaly`
- Compare `anomaly_raw_auto` vs `anomaly_raw_informed` — decide which contamination
  setting to carry into the dashboard

Next notebook: `05_risk_scoring_evaluation.ipynb`